# archiveWinston

### Modules

In [ ]:
from obspy import UTCDateTime
import obspy
from obspy.clients.fdsn import Client
from obspy.core.trace import Trace
from obspy.core.stream import Stream
from obspy import read
import numpy as np
from numpy import dtype
from scipy import stats

In [ ]:
import timeutils

import seismology.stream
import seismology.stream.datasource
from seismology.stream.datasource import createClient
from seismology.stream.nslcobject import str2nslc, setNSLC
from seismology.stream import removeWinstonGaps

from ioutils.filestructure import write2sds

In [ ]:
#nslc_list = 'VG.PSAG.00.EHZ'
#type(nslc_list) is str

### Input Parameters

In [ ]:
ds             = 'wws:vdap.org:16024'  # str : file:/file/structure/ or wws:waveserverip:port

nslc_list     = ['VG.PSAG.00.EHZ', 'VG.TMKS.00.EHZ']    # list or str : Stations to download (a single station may be entered as a str)
nslc_order    = 'nslc'                # str : Station code order : 'NSLC', 'SCNL', 'SCN'
nslc_out_list = ['VG.PSAG.01.EHZ', 'VG.PSAG.01.EHZ']    # list or str : Replacement station code for output files
#net, sta, loc, cha = 'VG', 'PSAG', '00', 'EHZ'

tstart        = '2017/09/18 11:02:06' # date format understood by ObsPy
tend          = '2017/09/18 11:03:46' # date format understand by ObsPy
tstart        = '2017/09/01 00:00:00' # date format understood by ObsPy
tend          = '2017/09/18 00:00:00' # date format understand by ObsPy
tstart        = '2018/01/17 00:00:00' # date format understood by ObsPy
tend          = '2018/01/17 03:59:59.99' # date format understand by ObsPy


nsec          = 3600                  # int : Maximum amount of time to download at once (seconds)
nsecproc      = 86400                 # int : Maximum amount of time to process at once (seconds)
prsc          = 0.01                  # float : Preciscion for createTimeChunks

#removeWWSgaps = True                  # bool : If True, removes -2**31 from all traces. Warning, False may cause errors when reading from WWS
verbose       = True

### Load, Process, Save

In [ ]:
tstart = UTCDateTime(tstart)
tend = UTCDateTime(tend)
dt1 = tstart
dt2 = tend
client = createClient(ds)
net, sta, loc, cha = str2nslc(nslc_list[0])
stmp = client.get_waveforms(net, sta, loc, cha, dt1, dt2)
stmp = removeWinstonGaps(stmp)
stmp = stmp.merge(method=1, fill_value=0)
stmp = setNSLC( stmp, nslc_out_list[0])
filepath  = write2sds(stmp, basedir='/Users/jjw2/Downloads/')

In [ ]:
# Establish datasource
if verbose: print('>>> Loading data from {}'.format(ds))
client = createClient(ds)

# Assert NSLC as list
if type(nslc_list) is str: nslc_list = [nslc_list]

# Assert tstart, tend as UTCDateTime
tstart = UTCDateTime(tstart)
tend = UTCDateTime(tend)

if nslc_out_list is None: nslc_out_list = nslc_list

# Loop through list of NSLCs
for nslc, nslc_out in zip(nslc_list, nslc_out_list):

    if verbose: print('>>> Station {}'.format(nslc))
    network, station, channel, location = str2nslc( nslc )

    # Loop through time chunks for output file
    proctstarts, proctends = timeutils.createTimeChunks(tstart, tend, nsec=nsecproc, prsc=0.01, verbose=False)
    for pt1, pt2 in zip(proctstarts, proctends):
        if verbose: print('>>> Prepping data for file storage from {} to {}'.format(pt1, pt2))

        st = Stream()

        # Create smaller time chunks to download, if necessary
        dtstarts, dtends = timeutils.createTimeChunks(pt1, pt2, nsec=nsec, prsc=0.01, verbose=False)
        for dt1, dt2 in zip(dtstarts, dtends):
            if verbose: print('>>> Downloading data from {} to {}'.format(dt1, dt2))

            try:

                stmp = client.get_waveforms(net, sta, loc, cha, dt1, dt2)
                stmp = removeWinstonGaps(stmp) # Loops through Streams
            
            except:
            
                stmp = Stream()

            st += stmp

        st = st.merge(method=1, fill_value=0)
        st = setNSLC( st, nslc_out)
        
        # Write Stream to File
        print(st)
        f = write2sds(st, basedir='/Users/jjw2/Downloads')
        if verbose: print('>>> File archived : {}'.format(f))
